# NLP Tasks with Large Language Models

Large Language Models (LLMs) can perform a wide range of **Natural Language Processing tasks** in a zero-shot or few-shot setting — without any task-specific training — simply by describing the task in the prompt. This approach, known as *prompting*, has become one of the most practical ways to build NLP systems today.

In this notebook you will explore how LLMs can be applied to core NLP tasks and learn how to interact with them programmatically.

## How we connect to LLMs: vLLM

We will use **[vLLM](https://docs.vllm.ai/)** as our inference backend. vLLM is a high-throughput engine for serving LLMs that uses *PagedAttention* to efficiently manage GPU memory, enabling fast responses even under concurrent load. It exposes an **OpenAI-compatible REST API**, so we can interact with it using the standard `openai` Python client — just pointing to a different server URL.

## What you will do

1. Connect to a hosted vLLM server and explore the API
2. Apply LLMs to classic **NLP tasks**:
   - Text summarization
   - Sentiment analysis
   - Named Entity Recognition (NER)
   - Machine translation
   - Question answering
   - Text classification
   - Prompt-based output length control
   - Query rewriting
3. Use **structured output** (JSON mode) for reliable extraction
4. Use **streaming** for interactive applications
5. Understand how **generation parameters** affect output quality
6. Complete **lab exercises** that combine multiple tasks

**References:**
- https://docs.vllm.ai/en/latest/
- https://docs.vllm.ai/en/latest/serving/openai_compatible_server.html
- https://www.promptingguide.ai/

In [ ]:
#Make sure your environment is activated before running this command
!pip install openai

---
## Setup: Connecting to the vLLM Server

To interact with the LLMs, we need to connect to the vLLM server. The OpenAI-compatible API requires three things:
- **`base_url`**: the URL of the vLLM server (provided by the course)
- **`api_key`**: an API key (provided by the course)
- **`model`**: the model to use — we can query the server to see what is available

In [ ]:
from openai import OpenAI

# TODO: fill in the server details
VLLM_BASE_URL = "https://amalia.novasearch.org/vlm/v1"  # replace with your vLLM server URL
VLLM_API_KEY  = "amalia012026"  # can be any non-empty string if auth is not enforced

client = OpenAI(
    base_url=VLLM_BASE_URL,
    api_key=VLLM_API_KEY,
)

In [ ]:
# List available models on this vLLM server
models = client.models.list()
for m in models.data:
    print(m.id)

# TODO: set this to the model you want to use
MODEL = models.data[0].id
print(f"\nUsing model: {MODEL}")

---
## Part 1: Core API Interactions

### 1.1 — Text completion (raw generation)

The `/v1/completions` endpoint takes a raw text prompt and continues it. This is the lowest-level interface — no chat template is applied.

In [ ]:
response = client.completions.create(
    model=MODEL,
    prompt="Natural Language Processing is a field of AI that",
    max_tokens=20,
    temperature=0.7,
)

print(response.choices[0].text)

### 1.2 — Chat completion (instruction-following)

The `/v1/chat/completions` endpoint applies the model's **chat template** (system + user + assistant turns). This is the recommended interface for instruction-tuned models.

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user",   "content": "Explain what vLLM is in two sentences."},
    ],
    max_tokens=200,
    temperature=0.3,
)

print(response.choices[0].message.content)

### 1.3 — Multi-turn conversation

To maintain context across turns, accumulate the message list and append each assistant reply before sending the next user message.

In [ ]:
history = [
    {"role": "system", "content": "You are an NLP tutor. Give concise, accurate answers."},
]

def chat(user_message: str) -> str:
    history.append({"role": "user", "content": user_message})
    response = client.chat.completions.create(
        model=MODEL,
        messages=history,
        max_tokens=300,
        temperature=0.5,
    )
    assistant_reply = response.choices[0].message.content
    history.append({"role": "assistant", "content": assistant_reply})
    return assistant_reply

print(chat("What is tokenization?"))
print()
print(chat("Can you give a concrete example in Python?"))

---
## Part 2: NLP Tasks via Prompting

LLMs can perform many classic NLP tasks **zero-shot** — just by describing the task in the prompt. This section demonstrates the most common ones.

We will use a helper function to keep the examples clean:

In [ ]:
def ask(system_prompt: str, user_prompt: str, temperature: float = 0.2) -> str:
    """Single-turn chat helper."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        max_tokens=512,
        temperature=temperature,
    )
    return response.choices[0].message.content

### 2.1 — Text Summarization

Summarization condenses a long document into a shorter representation while preserving the key information.

In [ ]:
article = """
Transformer models have revolutionized natural language processing since their introduction in the
2017 paper "Attention is All You Need" by Vaswani et al. Unlike recurrent networks, transformers
process all tokens in parallel using a self-attention mechanism, which allows them to capture
long-range dependencies much more efficiently. This architectural shift enabled the training of
much larger models such as BERT, GPT-2, and eventually GPT-4 and LLaMA. These large language
models have demonstrated emergent capabilities including in-context learning, chain-of-thought
reasoning, and instruction following, fundamentally changing how NLP systems are built and
deployed in production.
"""

summary = ask(
    system_prompt="You are an expert at summarizing scientific text. Be concise.",
    user_prompt=f"Summarize the following paragraph in one sentence:\n\n{article}",
)

print("Summary:", summary)

**Exercise:** Try different summarization styles:
- `"Summarize in bullet points"`
- `"Write a 3-sentence abstract"`
- `"Explain this to a 10-year-old"`

In [ ]:
# Your code here


### 2.2 — Sentiment Analysis

Sentiment analysis determines the emotional tone of a piece of text.

In [ ]:
reviews = [
    # English reviews
    "The hotel was absolutely stunning — the staff went above and beyond!",
    "Terrible experience. The room was dirty and the service was slow.",
    # European Portuguese reviews
    "O restaurante foi uma experiência incrível! A comida estava deliciosa e o serviço foi impecável.",
    "Muito desiludido com a encomenda. O produto chegou danificado e o apoio ao cliente foi horrível.",
    "O filme estava bem realizado, mas a história era um pouco previsível. Valeu a pena ver uma vez.",
]

system_prompt = (
    "You are a sentiment classifier. "
    "For each review, reply with exactly one word: Positive, Negative, or Neutral."
)

for review in reviews:
    label = ask(system_prompt, review, temperature=0.0)
    print(f"[{label.strip():8s}] {review}")

**Exercise:** Extend the prompt to also output a confidence score (0–1) and the main reason for the classification.

In [ ]:
# Your code here


### 2.3 — Named Entity Recognition (NER)

NER identifies and classifies named entities (people, organizations, locations, dates, etc.) in text.

In [ ]:
text = (
    "Elon Musk, CEO of Tesla and SpaceX, announced on Tuesday that the company "
    "plans to open a new Gigafactory in Berlin, Germany by mid-2026. "
    "The investment is estimated at €4 billion."
)

ner_prompt = f"""
Extract all named entities from the text below.
For each entity, list: entity text, entity type (PERSON, ORG, LOC, DATE, MONEY).
Format: one entity per line as "entity | TYPE".

Text: {text}
"""

result = ask(
    system_prompt="You are a Named Entity Recognition system. Extract entities precisely.",
    user_prompt=ner_prompt,
    temperature=0.0,
)

print(result)

**Exercise:** Try the same prompt on a sentence about a scientific paper or a news headline from a different domain.

In [ ]:
# Your code here


### 2.4 — Machine Translation

Modern LLMs are highly multilingual and can translate between dozens of languages.

In [ ]:
source_text = (
    "Large language models have demonstrated remarkable capabilities across a wide range "
    "of natural language processing tasks, often surpassing specialized models."
)

target_languages = ["Portuguese", "French", "German", "Japanese"]

for lang in target_languages:
    translation = ask(
        system_prompt=f"Translate the following English text to {lang}. Output only the translation.",
        user_prompt=source_text,
        temperature=0.1,
    )
    print(f"[{lang}] {translation}")
    print()

### 2.5 — Extractive Question Answering

Question answering (QA) asks the model to find an answer to a question based on a provided context passage.

In [ ]:
context = """
The attention mechanism was first introduced in the context of neural machine translation by
Bahdanau et al. in 2015. It allows the model to focus on different parts of the input sequence
when generating each output token. The Transformer architecture, proposed in 2017 by Vaswani
et al., extended this idea with multi-head self-attention, enabling the model to attend to
multiple representation subspaces simultaneously. This architecture became the foundation for
models like BERT (2018) and GPT-2 (2019).
"""

questions = [
    "Who introduced the attention mechanism?",
    "What does multi-head self-attention enable?",
    "In what year was the Transformer architecture proposed?",
    "Quem introduziu o mecanismo de atenção?",  # Portuguese question
]

system_prompt = (
    "Answer the question based only on the provided context. "
    "If the answer is not in the context, say 'Not found'."
)

for q in questions:
    user_prompt = f"Context:\n{context}\n\nQuestion: {q}"
    answer = ask(system_prompt, user_prompt, temperature=0.0)
    print(f"Q: {q}")
    print(f"A: {answer.strip()}")
    print()

### 2.6 — Text Classification

Beyond sentiment, LLMs can classify text into arbitrary custom categories defined in the prompt.

In [ ]:
categories = ["Sports", "Technology", "Politics", "Health", "Entertainment"]

headlines = [
    "Scientists develop a new vaccine against malaria using mRNA technology.",
    "Real Madrid wins the Champions League in a dramatic penalty shootout.",
    "Parliament votes on new data privacy bill affecting tech companies.",
    "The latest iPhone model features an AI chip for on-device processing.",
    "Oscar nominations announced: drama film leads with 10 nominations.",
]

system_prompt = (
    f"Classify the news headline into exactly one of these categories: {', '.join(categories)}. "
    "Reply with only the category name."
)

for headline in headlines:
    category = ask(system_prompt, headline, temperature=0.0)
    print(f"[{category.strip():14s}] {headline}")

### 2.7 — Prompt-Based Output Length Control

Beyond setting `max_tokens`, you can control verbosity **via the prompt itself**. This is useful when you want the model to follow a specific format (e.g., one sentence, bullet points, a word count limit) regardless of what `max_tokens` allows.

Note the difference:
- `max_tokens` is a **hard ceiling** on the number of tokens generated (the model will stop there)
- Prompt instructions are a **soft constraint** — the model tries to follow them, but they guide style and length in a more flexible way

**Exercise:** Try the same with an audience constraint instead of a length constraint — e.g., `"explain to a 10-year-old"` vs `"explain to a domain expert"`. How does the output change?

In [ ]:
text = (
    "Artificial intelligence is transforming many industries. "
    "From healthcare to finance, AI systems are being deployed to automate tasks, "
    "improve decision-making, and personalize services at scale."
)

length_instructions = [
    "in one sentence",
    "in under 30 words",
    "in exactly three bullet points",
    "as a detailed paragraph of at least 100 words",
]

for instruction in length_instructions:
    result = ask(
        system_prompt="You are a precise writing assistant. Follow the length instruction exactly.",
        user_prompt=f"Summarize the following text {instruction}:\n\n{text}",
        temperature=0.3,
    )
    print(f"[{instruction}]")
    print(result.strip())
    print()

### 2.8 — Query Rewriting

Query rewriting is an important NLP task in information retrieval and conversational AI. Given an ambiguous, verbose, or poorly-formed query, the model rewrites it into a cleaner version that is more likely to retrieve relevant results.

This is commonly used in **RAG (Retrieval-Augmented Generation)** pipelines to improve document retrieval before passing context to a generation model.

In [ ]:
queries = [
    # English queries
    "What was the neolithic revolution? Why did it start?",
    "how does attention work in transformers?",
    # European Portuguese queries
    "o que é o processamento de linguagem natural?",
    "como funciona o mecanismo de atenção nos transformers?",
    "Quais as melhores técnicas para treinar modelos de linguagem?",
]

system_prompt = (
    "You are a search query optimizer. "
    "Rewrite the user's query to be clearer, more specific, and better suited for a search engine. "
    "Keep the rewritten query in the same language as the original. "
    "Output only the rewritten query, nothing else."
)

for query in queries:
    rewritten = ask(system_prompt, query, temperature=0.2)
    print(f"Original : {query}")
    print(f"Rewritten: {rewritten.strip()}")
    print()

**Exercise:** Apply query rewriting to a question from a different domain (e.g., medicine, history). Does the rewritten version return better results when you imagine searching for it?

In [ ]:
# Your code here


---
## Part 3: Structured Output (JSON Mode)

When integrating LLMs into applications, you often need the output in a **structured format** (e.g., JSON) that can be parsed programmatically. vLLM supports guided decoding via the `response_format` parameter.

In [ ]:
import json

review = "The laptop is incredibly fast and the battery lasts all day, but the keyboard feels cheap."

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": (
                "You are a product review analyzer. "
                "Extract structured information from the review and return valid JSON with keys: "
                "'sentiment' (Positive/Negative/Mixed), "
                "'pros' (list of strings), "
                "'cons' (list of strings), "
                "'score' (integer 1-5)."
            ),
        },
        {"role": "user", "content": review},
    ],
    response_format={"type": "json_object"},
    max_tokens=256,
    temperature=0.0,
)

result = json.loads(response.choices[0].message.content)
print(json.dumps(result, indent=2))

**Exercise:** Parse the JSON result above and display the pros and cons in a readable format using print statements.

In [ ]:
# Your code here


---
## Part 4: Streaming Responses

For long generations, **streaming** lets you process tokens as they arrive instead of waiting for the full response. This is essential for building responsive chat interfaces.

In [ ]:
import sys

stream = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are an NLP expert."},
        {"role": "user",   "content": "Explain the difference between BERT and GPT architectures."},
    ],
    max_tokens=400,
    temperature=0.5,
    stream=True,
)

for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="", flush=True)

print()  # newline at end

---
## Part 5: Generation Parameters

The quality and style of LLM output is strongly influenced by **decoding parameters**. Experiment with the cells below to understand their effect.

| Parameter | Effect |
|-----------|--------|
| `temperature` | Controls randomness. 0 = greedy (deterministic), >1 = more creative/chaotic |
| `top_p` | Nucleus sampling: only sample from the top-p probability mass |
| `top_k` | Only sample from the top-k most likely tokens |
| `max_tokens` | Maximum number of tokens to generate |
| `presence_penalty` | Penalizes tokens that have appeared in the context (promotes topic diversity) |
| `frequency_penalty` | Penalizes tokens proportional to how often they have appeared (reduces repetition) |

In [ ]:
prompt = "Write a one-sentence creative description of the ocean."

for temp in [0.0, 0.5, 1.0, 1.5]:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=80,
        temperature=temp,
    )
    text = response.choices[0].message.content.strip()
    print(f"[temperature={temp}] {text}")
    print()

---
## Lab Exercises

The following exercises ask you to combine and extend the techniques from this notebook. Each one requires you to write code from scratch (using the helpers already defined), design prompts carefully, and think about what the model is doing.

Work through them in order — later exercises build on earlier ones.

### Exercise 1: Closed-Book Question Answering with Self-Assessment

In extractive QA (Part 2.5) the model could only use the provided context. Here the model answers **from its own knowledge** — and also rates its own confidence.

**Requirements:**
1. Choose a topic you know well (history, science, literature, sports, etc.) and write **5 factual questions** about it
2. For each question, ask the LLM to:
   - Answer the question
   - Give a confidence score from 0 to 1 (how sure is it the answer is correct?)
   - Briefly justify the confidence score
3. Return the result as JSON with keys `"answer"`, `"confidence"`, and `"justification"`
4. Print results in a readable format and identify which answers the model is least confident about

**Hints:**
- Use `response_format={"type": "json_object"}` and parse with `json.loads()`
- Use `temperature=0.0` for more consistent, less hallucinated answers
- Try a mix of easy and hard questions and compare the confidence scores
- You can write questions in Portuguese — e.g., about Portuguese history or culture:
  - *"Em que ano foi assinado o Tratado de Lisboa?"*
  - *"Quem escreveu 'Os Lusíadas' e em que século?"*
  - *"Qual é a capital do distrito de Évora?"*

In [ ]:
# Exercise 1 — your code here


### Exercise 2: Multi-Task Text Analysis Pipeline

Build a pipeline that runs **three NLP tasks in sequence** on the same input text.

**Requirements:**
1. Choose any short article or paragraph (2–4 sentences, any topic you like)
2. For each text, run the following three tasks and print the results clearly:
   - **Summarization**: one sentence summary
   - **Topic classification**: classify into one of `[Science, Politics, Sports, Technology, Culture, Other]`
   - **NER**: extract all named entities with their types
3. Wrap the three steps into a single function `analyze(text: str)` that returns a dictionary with keys `"summary"`, `"topic"`, and `"entities"`

**Hints:**
- Reuse the `ask()` helper from Part 2
- For NER, ask the model to return `entity | TYPE` lines and parse them into a list of `(entity, type)` tuples
- Run the pipeline on at least **2 different texts** to compare results
- You can use Portuguese texts — some samples to get you started:

```python
texts = [
    # Portuguese texts
    ("Cristiano Ronaldo, natural de Funchal, na ilha da Madeira, iniciou a sua carreira no Sporting CP "
     "antes de se transferir para o Manchester United em 2003. É considerado um dos melhores "
     "futebolistas de todos os tempos, tendo vencido cinco Bolas de Ouro."),

    ("A Fundação Champalimaud, situada em Lisboa junto ao Tejo, é um dos principais centros de "
     "investigação em neurociências e oncologia na Europa. Foi inaugurada em 2010 e conta com "
     "cientistas de mais de 30 países."),

    # English text
    ("The European Space Agency launched its Hera mission in October 2024 to study the aftermath "
     "of NASA's DART spacecraft impact on the asteroid Dimorphos, aiming to develop planetary "
     "defence strategies against potential asteroid threats to Earth."),
]
```

In [ ]:
# Exercise 2 — your code here


### Exercise 3: Domain-Specific Chatbot

Build a simple multi-turn chatbot specialised for a domain of your choice (e.g., a cooking assistant, a Python tutor, a travel advisor).

**Requirements:**
1. Define a clear system prompt that gives the bot its persona and constraints (e.g., "only answer questions about cooking, politely decline off-topic questions")
2. Use the multi-turn pattern from Part 1.3 to hold a conversation of **at least 4 turns**
3. At some point ask a question that is *off-topic* and observe how the bot responds
4. Print each turn clearly labelled `User:` / `Assistant:`

**Hints:**
- Reuse the `chat()` function from section 1.3, but reset `history` with your new system prompt first
- A tight, specific system prompt leads to more consistent behaviour
- **Portuguese idea:** build a chatbot that only communicates in European Portuguese and acts as a guide to visiting Lisbon — ask it about neighbourhoods, food, transport, and then try asking something completely unrelated

In [ ]:
# Exercise 5 — your code here
